In [56]:
import os
import torch
from tqdm import tqdm
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [57]:
print("Torch version: ", torch.__version__)
print("Numpy version: ", np.__version__)
print("Pandas version: ", pd.__version__)
print("Seaborn version: ", sns.__version__)
# print("TQDM version: ", tqdm.__version__)

Torch version:  2.13.0
Numpy version:  2.5.1
Pandas version:  3.0.3
Seaborn version:  0.13.2


In [58]:
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
else:
  device = torch.device("cpu")
  
print(f"Running on device: {device}")

Running on device: mps


In [59]:
images = os.listdir("all_data")
print(f"Number of images: {len(images)}")

Number of images: 5676


In [60]:
class ImageDataset(Dataset):
  def __init__(self, img_dir, csv_file, transforms=None):
    """
    Args:
      img_dir (string): Path to the image folder directory.
      csv_file (string): Path to the csv file containing image ids and labels.
      transforms (callable, optional): Optional transform to be applied to a sample.
    """
    # load the CSV into a pandas dataframe
    self.image_labels = pd.read_csv(csv_file)
    self.img_dir = img_dir
    self.transforms = transforms
  
  def __len__(self):
    return len(self.image_labels) 
  
  def __getitem__(self, row_index):
    # Get the image file name from CSV
    img_name = self.image_labels.iloc[row_index, 0]
    img_path = os.path.join(self.img_dir, img_name + ".JPG")
    
    # load the image using PIL
    image = Image.open(img_path)
    
    # get the corresponding image label
    label = self.image_labels.iloc[row_index, 1]
    
    if self.transforms:
      image = self.transforms(image)
    
    return image, label

In [61]:
batch_size = 32

my_transforms = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor()
])

train_dataset = ImageDataset(
  img_dir="all_data",
  csv_file="train_ids_labels.csv",
  transforms=my_transforms
)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [62]:
batch_image, batch_label = next(iter(train_dataloader))
# torch.tensor(batch_image).shape, torch.tensor(batch_label).shape
batch_image.shape, batch_label.shape # already tensors from ToTensor()

(torch.Size([32, 3, 224, 224]), torch.Size([32]))

In [64]:
from collections import Counter

def class_counts(dataset):
  c = Counter(x[1] for x in tqdm(dataset))
  class_to_index = dataset.class_to_idx
  return pd.Series({cat: c[idx] for cat, idx in class_to_index.items()})

class_counts(train_dataset)

100%|██████████| 4026/4026 [00:39<00:00, 102.19it/s]


AttributeError: 'ImageDataset' object has no attribute 'class_to_idx'